In [1]:
import sqlite3
import struct
import time
from pathlib import Path

DB_PATH : str = "database/app.db"   # database file, created next to the notebook
RESET : bool = False        # True deletes the database first (cold start)

# Database development
This notebook describes the step by step process of the developement of the application database. 

## Overview
The database is an SQLite database, residing on disk. We chose SQLite because of:
- Fits the laptop constraint. The README requires a modest footprint on the professor’s laptop. SQLite is a single file with no server to install or run.
Nothing extra to set up. Python’s built-in sqlite3 module is enough, so teammates can clone the repo and run it.
- Matches your data size. The cache is bounded, so you’ll have thousands to tens of thousands of rows. That is well within what SQLite handles.
- Fast local lookups. There’s no network hop, so an exact cache hit takes well under a millisecond, which helps the sub-second target.
- Easy resets for benchmarks. A cold start is deleting the file, and a warm start is copying one. That makes cold-versus-warm comparisons repeatable.
- Vector search is covered. sqlite-vec provides it, with NumPy over blob columns as a fallback. At your scale, brute-force search takes milliseconds.
- Keyword search is built in. FTS5 can index evidence passages for hybrid retrieval at no extra cost.
- Concurrency is manageable. WAL mode plus a single writer queue fits your bounded-queue design, so readers and the writer don’t block each other.
- Simple to ship. The demo and the benchmark data are one file.
- Portable if you outgrow it. If all database access goes through a small interface, you could move to Postgres + pgvector later without changing the other workstream

The database will follow this schema:
https://dbdiagram.io/d/Fact-check-DB-6ad76e2cf65bbc60f6de9441

## 1. Schema
**Conventions**

- Timestamps are `INTEGER` values in epoch milliseconds, so latency is a subtraction (`verdict_at - stable_at`).
- Booleans are `INTEGER` values: `0` for false, `1` for true.
- Embeddings are `BLOB` values holding float32 arrays. All vectors must come from the same embedding model.
- JSON is stored as `TEXT`.

In [2]:
SCHEMA_SOURCES = """
CREATE TABLE IF NOT EXISTS sources (
  id          INTEGER PRIMARY KEY,
  source_url  TEXT    NOT NULL,
  status      TEXT    NOT NULL DEFAULT 'queued'
              CHECK (status IN ('queued','transcribing','processing','done','failed')),
  created_at  INTEGER NOT NULL
              DEFAULT (CAST((julianday('now') - 2440587.5) * 86400000 AS INTEGER))
) STRICT;
"""

In [3]:
SCHEMA_CACHES = """
-- Cache 1: verified claims (reusable verdicts)
CREATE TABLE IF NOT EXISTS verified_claims (
  id             INTEGER PRIMARY KEY,
  claim_text     TEXT    NOT NULL,
  embedding      BLOB,
  entities       TEXT,                       -- JSON
  numbers        TEXT,                       -- JSON
  polarity       INTEGER NOT NULL DEFAULT 1  -- 1 assertion, 0 negation
                 CHECK (polarity IN (0,1)),
  verdict        TEXT    NOT NULL
                 CHECK (verdict IN ('supported','contradicted','uncertain')),
  explanation    TEXT,
  verified_at    INTEGER NOT NULL,
  expires_at     INTEGER NOT NULL,
  model_version  TEXT    NOT NULL,
  last_used_at   INTEGER,
  hit_count      INTEGER NOT NULL DEFAULT 0
) STRICT;

-- Cache 2: evidence passages (the RAG index)
CREATE TABLE IF NOT EXISTS evidence (
  id             INTEGER PRIMARY KEY,
  evidence_text  TEXT    NOT NULL,
  source_url     TEXT    NOT NULL,
  source_date    TEXT,
  embedding      BLOB,
  fetched_at     INTEGER NOT NULL,
  last_used_at   INTEGER
) STRICT;

-- Which passages back which verdict
CREATE TABLE IF NOT EXISTS verdict_evidence (
  verified_claim_id  INTEGER NOT NULL
                     REFERENCES verified_claims(id) ON DELETE CASCADE,
  evidence_id        INTEGER NOT NULL
                     REFERENCES evidence(id) ON DELETE CASCADE,
  stance             TEXT    NOT NULL
                     CHECK (stance IN ('supports','contradicts','neutral')),
  PRIMARY KEY (verified_claim_id, evidence_id)
) STRICT;
"""

In [4]:
SCHEMA_RUN = """
CREATE TABLE IF NOT EXISTS transcript_segments (
  id         INTEGER PRIMARY KEY,
  video_id   INTEGER NOT NULL REFERENCES sources(id) ON DELETE CASCADE,
  start_ms   INTEGER NOT NULL,
  end_ms     INTEGER NOT NULL CHECK (end_ms >= start_ms),
  text       TEXT    NOT NULL,
  is_stable  INTEGER NOT NULL DEFAULT 0 CHECK (is_stable IN (0,1))
) STRICT;

CREATE TABLE IF NOT EXISTS claims (
  id                 INTEGER PRIMARY KEY,
  video_id           INTEGER NOT NULL REFERENCES sources(id) ON DELETE CASCADE,
  verified_claim_id  INTEGER REFERENCES verified_claims(id) ON DELETE SET NULL,
  claim_text         TEXT    NOT NULL,
  start_ms           INTEGER NOT NULL,
  end_ms             INTEGER NOT NULL CHECK (end_ms >= start_ms),
  status             TEXT    NOT NULL DEFAULT 'not_checked'
                     CHECK (status IN ('not_checked','checking','supported',
                                       'contradicted','uncertain')),
  path               TEXT
                     CHECK (path IS NULL OR
                            path IN ('fresh','claims_cache','evidence_cache')),
  stable_at          INTEGER NOT NULL,   -- latency stopwatch starts
  verdict_at         INTEGER             -- latency stopwatch stops
) STRICT;

CREATE TABLE IF NOT EXISTS claim_segments (
  claim_id    INTEGER NOT NULL REFERENCES claims(id) ON DELETE CASCADE,
  segment_id  INTEGER NOT NULL REFERENCES transcript_segments(id) ON DELETE CASCADE,
  PRIMARY KEY (claim_id, segment_id)
) STRICT;
"""

In [5]:
SCHEMA_INDEXES = """
CREATE INDEX IF NOT EXISTS idx_segments_video
  ON transcript_segments (video_id, start_ms);
CREATE INDEX IF NOT EXISTS idx_claims_video
  ON claims (video_id, start_ms);
CREATE INDEX IF NOT EXISTS idx_claims_status
  ON claims (status);
CREATE INDEX IF NOT EXISTS idx_claims_verified
  ON claims (verified_claim_id);
CREATE INDEX IF NOT EXISTS idx_claim_segments_segment
  ON claim_segments (segment_id);
CREATE INDEX IF NOT EXISTS idx_verified_expires
  ON verified_claims (expires_at);
CREATE INDEX IF NOT EXISTS idx_verified_lru
  ON verified_claims (last_used_at);
CREATE INDEX IF NOT EXISTS idx_evidence_lru
  ON evidence (last_used_at);
CREATE INDEX IF NOT EXISTS idx_evidence_url
  ON evidence (source_url);
CREATE INDEX IF NOT EXISTS idx_verdict_evidence_evidence
  ON verdict_evidence (evidence_id);
"""

SCHEMA = "\n".join([SCHEMA_SOURCES, SCHEMA_CACHES, SCHEMA_RUN, SCHEMA_INDEXES])

## 2. Helper functions

`connect()` opens the database with four `PRAGMA` settings:

- `foreign_keys = ON` enforces foreign keys, which SQLite leaves off by default.
- `journal_mode = WAL` lets readers (the GUI) work while the writer saves new rows.
- `synchronous = NORMAL` makes commits faster and is safe in WAL mode.
- `busy_timeout = 5000` waits up to 5 seconds for a lock instead of failing at once.

`foreign_keys`, `synchronous` and `busy_timeout` reset on every new connection, so everything in the project should open the database through `connect()`.

The other helpers are the shared clock (`now_ms`) and the conversion between a list of floats and the bytes stored in the `embedding` columns.

In [6]:
def connect(path: str = DB_PATH) -> sqlite3.Connection:
    """Open the database with the settings every connection should use."""
    con = sqlite3.connect(path, timeout=5)
    con.execute("PRAGMA foreign_keys = ON")
    con.execute("PRAGMA journal_mode = WAL")
    con.execute("PRAGMA synchronous = NORMAL")
    con.execute("PRAGMA busy_timeout = 5000")
    return con


def create_schema(con: sqlite3.Connection) -> None:
    schema = SCHEMA
    if sqlite3.sqlite_version_info < (3, 37, 0):
        # STRICT tables need SQLite 3.37 or newer
        schema = schema.replace(") STRICT;", ");")
    con.executescript(schema)
    con.commit()


def try_load_sqlite_vec(con: sqlite3.Connection) -> bool:
    """Load the optional sqlite-vec extension. Returns False if unavailable."""
    try:
        import sqlite_vec

        con.enable_load_extension(True)
        sqlite_vec.load(con)
        con.enable_load_extension(False)
        return True
    except (ImportError, AttributeError, sqlite3.OperationalError):
        return False


def now_ms() -> int:
    """The one clock every workstream should use for timestamps."""
    return time.time_ns() // 1_000_000


def vector_to_blob(vec) -> bytes:
    """Pack a list of floats as float32 bytes (what sqlite-vec and NumPy read)."""
    return struct.pack(f"{len(vec)}f", *vec)


def blob_to_vector(blob: bytes) -> list:
    return list(struct.unpack(f"{len(blob) // 4}f", blob))

## 3. Create the database

In [7]:
try:
    con.close()          # close a connection left over from a previous run
except NameError:
    pass

if RESET:
    for suffix in ("", "-wal", "-shm"): # Unlink database files for deletion
        Path(DB_PATH + suffix).unlink(missing_ok=True)

con = connect(DB_PATH)
create_schema(con)

tables = [r[0] for r in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
print(f"SQLite {sqlite3.sqlite_version}, database: {DB_PATH}")
print("sqlite-vec available:", try_load_sqlite_vec(con))
print("Tables:", ", ".join(tables))

for table in tables:
    print(f"\n{table}")
    for cid, name, ctype, notnull, default, pk in con.execute(f"PRAGMA table_info({table})"):
        flags = []
        if pk:
            flags.append("PK")
        if notnull:
            flags.append("NOT NULL")
        print(f"  {name:<20} {ctype:<8} {' '.join(flags)}")

SQLite 3.43.1, database: database/app.db
sqlite-vec available: True
Tables: claim_segments, claims, evidence, sources, transcript_segments, verdict_evidence, verified_claims

claim_segments
  claim_id             INTEGER  PK NOT NULL
  segment_id           INTEGER  PK NOT NULL

claims
  id                   INTEGER  PK
  video_id             INTEGER  NOT NULL
  verified_claim_id    INTEGER  
  claim_text           TEXT     NOT NULL
  start_ms             INTEGER  NOT NULL
  end_ms               INTEGER  NOT NULL
  status               TEXT     NOT NULL
  path                 TEXT     
  stable_at            INTEGER  NOT NULL
  verdict_at           INTEGER  

evidence
  id                   INTEGER  PK
  evidence_text        TEXT     NOT NULL
  source_url           TEXT     NOT NULL
  source_date          TEXT     
  embedding            BLOB     
  fetched_at           INTEGER  NOT NULL
  last_used_at         INTEGER  

sources
  id                   INTEGER  PK
  source_url           

## 4. Sanity check

In [8]:
demo = connect(":memory:")
create_schema(demo)

# A claim goes through the pipeline
stable = now_ms()   # the latency stopwatch starts
demo.execute("INSERT INTO sources (source_url) VALUES ('video.mp4')")
demo.execute("""INSERT INTO transcript_segments (video_id, start_ms, end_ms, text, is_stable)
                VALUES (1, 0, 2000, 'hello', 1)""")
demo.execute("""INSERT INTO claims (video_id, claim_text, start_ms, end_ms, stable_at)
                VALUES (1, 'test claim', 0, 2000, ?)""", (stable,))
demo.execute("INSERT INTO claim_segments VALUES (1, 1)")
demo.execute("""INSERT INTO verified_claims
                (claim_text, embedding, verdict, verified_at, expires_at, model_version)
                VALUES ('test claim', ?, 'supported', ?, ?, 'm1')""",
             (vector_to_blob([0.1, 0.2, 0.3]), now_ms(), now_ms() + 1000))
demo.execute("""UPDATE claims
                SET verified_claim_id = 1, status = 'supported',
                    path = 'claims_cache', verdict_at = ?
                WHERE id = 1""", (stable + 40,))   # pretend the cache hit took 40 ms
demo.commit()

latency = demo.execute("SELECT verdict_at - stable_at FROM claims WHERE id = 1").fetchone()[0]
print("latency of the demo claim:", latency, "ms")

blob = demo.execute("SELECT embedding FROM verified_claims").fetchone()[0]
print("embedding round trip:", [round(x, 2) for x in blob_to_vector(blob)])

checks = [
    ("bad status value", "UPDATE claims SET status = 'nope' WHERE id = 1"),
    ("bad path value", "UPDATE claims SET path = 'web' WHERE id = 1"),
    ("missing source (foreign key)",
     "INSERT INTO claims (video_id, claim_text, start_ms, end_ms, stable_at) "
     "VALUES (99, 'x', 0, 1, 0)"),
    ("string in a timestamp column (STRICT)",
     "UPDATE claims SET stable_at = 'abc' WHERE id = 1"),
]

for label, sql in checks:
    try:
        demo.execute(sql)
        print(f"{label}: ACCEPTED (unexpected)")
    except sqlite3.IntegrityError:
        print(f"{label}: rejected")

# Deleting a source also deletes its segments and claims
demo.execute("DELETE FROM sources WHERE id = 1")
demo.commit()
left = demo.execute("SELECT COUNT(*) FROM claims").fetchone()[0]
print("claims left after deleting the source:", left)

demo.close()

latency of the demo claim: 40 ms
embedding round trip: [0.1, 0.2, 0.3]
bad status value: rejected
bad path value: rejected
missing source (foreign key): rejected
string in a timestamp column (STRICT): rejected
claims left after deleting the source: 0
